<a href="https://colab.research.google.com/github/SpaceBites/Illegal-forest-detection-/blob/main/Illegal_forest_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install rasterio numpy scipy matplotlib

In [ ]:
from google.colab import files
import zipfile
import os

# Upload your two zip files - a file picker will pop up
# Select BOTH before.zip and after.zip at the same time
uploaded = files.upload()

# Unzip both
for zip_name in uploaded.keys():
    with zipfile.ZipFile(zip_name, 'r') as z:
        z.extractall('/content/')
    print(f"Unzipped: {zip_name}")

# Set folder paths
BEFORE_FOLDER = "/content/before"
AFTER_FOLDER  = "/content/after"

# Verify files are there
print("\nBefore folder contents:", os.listdir(BEFORE_FOLDER))
print("After folder contents:",  os.listdir(AFTER_FOLDER))

In [ ]:
import os
import numpy as np
import rasterio
from scipy import ndimage
import matplotlib.pyplot as plt

# ---- Tunable parameters ----
NDVI_DROP_THRESHOLD = 0.30
MIN_FOREST_NDVI     = 0.40
MIN_PATCH_PIXELS    = 10


def load_single_band(path):
    with rasterio.open(path) as src:
        return src.read(1).astype(float), src.transform, src.crs


def compute_ndvi(red, nir):
    return (nir - red) / (nir + red + 1e-10)


def find_band_pairs(folder):
    """
    Scans a folder and pairs up B04 and B08 files by their shared location name.
    Expects filenames like:  Amazon_B04.tif / Amazon_B08.tif
    Returns dict: { 'Amazon': {'B04': '/path/B04.tif', 'B08': '/path/B08.tif'}, ... }
    """
    files = [f for f in os.listdir(folder) if f.endswith('.tif')]
    pairs = {}
    for f in files:
        name = f.replace('.tif', '')
        if 'B04' in name:
            location = name.replace('B04', '').strip('_- ')
            pairs.setdefault(location, {})['B04'] = os.path.join(folder, f)
        elif 'B08' in name:
            location = name.replace('B08', '').strip('_- ')
            pairs.setdefault(location, {})['B08'] = os.path.join(folder, f)
    return pairs


def detect_for_location(before_b04, before_b08, after_b04, after_b08):
    red1, transform, crs = load_single_band(before_b04)
    nir1, _, _           = load_single_band(before_b08)
    red2, _, _           = load_single_band(after_b04)
    nir2, _, _           = load_single_band(after_b08)

    ndvi_before = compute_ndvi(red1, nir1)
    ndvi_after  = compute_ndvi(red2, nir2)
    ndvi_diff   = ndvi_before - ndvi_after

    loss_mask = (ndvi_diff > NDVI_DROP_THRESHOLD) & (ndvi_before > MIN_FOREST_NDVI)
    labeled, num_patches = ndimage.label(loss_mask)
    pixel_area_m2 = abs(transform[0] * transform[4])

    results = []
    for patch_id in range(1, num_patches + 1):
        patch = labeled == patch_id
        n_pixels = patch.sum()
        if n_pixels < MIN_PATCH_PIXELS:
            continue
        area_ha    = round((n_pixels * pixel_area_m2) / 10000, 2)
        avg_drop   = ndvi_diff[patch].mean()
        confidence = round(min(99, (avg_drop / NDVI_DROP_THRESHOLD) * 65), 1)
        results.append({
            "area_hectares":     area_ha,
            "confidence_percent": confidence,
            "patch_id":          int(patch_id),
        })

    results.sort(key=lambda r: r["area_hectares"], reverse=True)
    return results, loss_mask, ndvi_before, ndvi_after


# ---- Run for all locations ----
before_pairs = find_band_pairs(BEFORE_FOLDER)
after_pairs  = find_band_pairs(AFTER_FOLDER)

all_locations = set(before_pairs.keys()) & set(after_pairs.keys())

if not all_locations:
    print("No matching location pairs found. Check your filenames.")
else:
    for location in sorted(all_locations):
        print(f"\n{'='*50}")
        print(f"  Location: {location}")
        print(f"{'='*50}")

        b = before_pairs[location]
        a = after_pairs[location]

        if 'B04' not in b or 'B08' not in b or 'B04' not in a or 'B08' not in a:
            print("  Missing B04 or B08 file — skipping.")
            continue

        results, loss_mask, ndvi_before, ndvi_after = detect_for_location(
            b['B04'], b['B08'], a['B04'], a['B08']
        )

        if not results:
            print("  No significant forest loss detected.")
        else:
            print(f"  Possible illegal logging — {len(results)} area(s) flagged")
            for r in results:
                print(f"    Area: {r['area_hectares']} ha   Confidence: {r['confidence_percent']}%")

        # Visualize
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        fig.suptitle(f"Location: {location}", fontsize=14, fontweight='bold')
        axes[0].imshow(ndvi_before, cmap="YlGn", vmin=-1, vmax=1)
        axes[0].set_title("NDVI — Before")
        axes[1].imshow(ndvi_after, cmap="YlGn", vmin=-1, vmax=1)
        axes[1].set_title("NDVI — After")
        axes[2].imshow(loss_mask, cmap="Reds")
        axes[2].set_title("Detected Forest Loss")
        for ax in axes: ax.axis("off")
        plt.tight_layout()
        plt.show()